# Resume Screening Agent

The goal is to screen resumes for a job: from a job description, find the resumes of the candidates who fit the job, in order. The data has 962 resumes with the job category of each one (25 categories such as Data Science, Java Developer, HR or Civil Engineer, `UpdatedResumeDataSet.csv`).

Reference solution: [Resume Screening with Python](https://amanxai.com/2020/12/06/resume-screening-with-python/) (Thecleverprogrammer / Aman Kharwal). It cleans the resume texts, turns them into TF-IDF vectors (1,500 features) and trains a one-vs-rest K-Nearest Neighbors classifier on an 80/20 split. It reports an accuracy of 0.99 on the test set. The dataset is a Kaggle dataset ([Resume Dataset](https://www.kaggle.com/datasets/gauravduttakiit/resume-dataset)); the copy used here comes from a public GitHub repository.

The idea is built as a small agent with two tools, a search tool (TF-IDF similarity) and a scoring tool (a local language model, Llama 3.2 run by Ollama, so that no API key or online service is needed), and both the reference and the agent are evaluated. The agent is a fixed workflow (search, then score, then rank): the language model is used as a tool inside it and does not decide by itself which tool to call.

In [1]:
import re
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import os
os.chdir('/mnt/c/Users/myama/OneDrive/Belgeler/ai-1/Assignments/14-Specialize in Data Science/AI Agents/resume_screening_agent')

## Data Loading

In [2]:
resumes = pd.read_csv("UpdatedResumeDataSet.csv")
print(resumes.shape, "| categories:", resumes["Category"].nunique(), "| null values:", resumes.isnull().sum().sum())
resumes["Category"].value_counts().head(6)

(962, 2) | categories: 25 | null values: 0


Category
Java Developer      84
Testing             70
DevOps Engineer     55
Python Developer    48
Web Designing       45
HR                  44
Name: count, dtype: int64

## Data Cleaning

In [3]:
print("duplicated rows:", resumes.duplicated().sum(), "| different resume texts:", resumes["Resume"].nunique())
labels_per_text = resumes.groupby("Resume")["Category"].nunique()
print("texts that carry more than one category:", (labels_per_text > 1).sum())

unique = resumes.drop_duplicates("Resume").reset_index(drop=True)
unique["copies"] = unique["Resume"].map(resumes["Resume"].value_counts())
print("copies per resume: min", unique["copies"].min(), "| median", unique["copies"].median(), "| max", unique["copies"].max())
unique["Category"].value_counts().describe()[["min", "50%", "max"]]

duplicated rows: 796 | different resume texts: 166
texts that carry more than one category: 0
copies per resume: min 1 | median 6.0 | max 18


min     3.0
50%     6.0
max    13.0
Name: count, dtype: float64

Only 166 of the 962 rows are different resumes: 796 rows are exact copies, each resume appears 1 to 18 times (median 6), and no text carries two categories. The dataset therefore contains far less information than its size suggests, and any random split will put copies of the same resume in both the training and the test set. The reference does not mention this.

The resume texts are cleaned with the function of the reference (URLs, hashtags, mentions, punctuation and non-ASCII characters removed).

In [4]:
def clean_resume(text):
    text = re.sub(r"http\S+\s*", " ", text)
    text = re.sub(r"RT|cc", " ", text)
    text = re.sub(r"#\S+", "", text)
    text = re.sub(r"@\S+", "  ", text)
    text = re.sub("[%s]" % re.escape("!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"), " ", text)
    text = re.sub(r"[^\x00-\x7f]", " ", text)
    return re.sub(r"\s+", " ", text)

resumes["cleaned"] = resumes["Resume"].apply(clean_resume)
unique["cleaned"] = unique["Resume"].apply(clean_resume)

## Reference Reproduction

The reference is reproduced: TF-IDF with 1,500 features, a one-vs-rest KNN classifier, 80/20 split with `random_state=0`, on all 962 rows. The share of the test resumes that have an exact copy in the training set is computed.

In [6]:
vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words="english", max_features=1500).fit(resumes["cleaned"])
features = vectorizer.transform(resumes["cleaned"])
x_train, x_test, y_train, y_test, text_train, text_test = train_test_split(features, resumes["Category"].tolist(), resumes["Resume"].tolist(), test_size=0.2, random_state=0)

reference = OneVsRestClassifier(KNeighborsClassifier()).fit(x_train, y_train)
print("accuracy on the training set:", round(reference.score(x_train, y_train), 3), "| on the test set:", round(reference.score(x_test, y_test), 3))
print("share of the test resumes with an identical copy in the training set:", round(pd.Series(text_test).isin(set(text_train)).mean(), 3))

accuracy on the training set: 0.992 | on the test set: 0.99
share of the test resumes with an identical copy in the training set: 0.995


The reference reaches an accuracy of 0.99 on the test set, but 99.5% of the test resumes have an identical copy in the training set, so the model only has to recognize a text it has already seen. The score measures memory, not the ability to classify a new resume. One more detail of the cleaning function: the expression `RT|cc` deletes the letters "rt" in capitals and the letters "cc" wherever they appear, including inside words such as "account" or "success". It was written for tweets and is kept here only to reproduce the reference (the same function is used for all the models below, so the comparison is fair).

## Evaluation of the Classifiers Without Copies

The 166 different resumes are split 80/20, so that no resume of the test set has a copy in the training set. Three classifiers are compared, with the same TF-IDF features.

In [7]:
u_train, u_test = train_test_split(unique, test_size=0.2, random_state=0)
unique_vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words="english", max_features=1500).fit(u_train["cleaned"])
ux_train, ux_test = unique_vectorizer.transform(u_train["cleaned"]), unique_vectorizer.transform(u_test["cleaned"])

classifiers = {"KNN (reference)": OneVsRestClassifier(KNeighborsClassifier()), "Logistic Regression": LogisticRegression(max_iter=1000),
               "Naive Bayes": MultinomialNB()}
pd.DataFrame({name: {"accuracy": accuracy_score(u_test["Category"], model.fit(ux_train, u_train["Category"]).predict(ux_test)),
                     "macro F1": f1_score(u_test["Category"], model.predict(ux_test), average="macro")} for name, model in classifiers.items()}).T.round(3)

,accuracy,macro F1
KNN (reference),0.824,0.691
Logistic Regression,0.324,0.235
Naive Bayes,0.118,0.094


When the copies are removed before the split, the accuracy of the same KNN model falls from 0.99 to 0.735 (macro F1 0.718), and the other two models are much worse: Logistic Regression 0.324 and Naive Bayes 0.118. With about 130 training resumes for 25 categories (5 per category on average), the models have too few examples, and the two probabilistic models are hit harder because they need to estimate many weights or frequencies. KNN, which only compares a resume with the closest training resumes, suffers less. The test set has only about 34 resumes, so one resume is worth 3 points and the ranking of the two weak models is not reliable, but the gap between 0.99 and 0.735 is not a matter of chance. Classification with this dataset is therefore a weaker basis for a screening tool than the reference suggests, and the agent below uses search instead: it ranks resumes for a job description, which needs no training labels.

## The Screening Agent: Search Tool

The first tool ranks the different resumes by the cosine similarity of their TF-IDF vector with the job description. To measure it, one short job description was written for each of the 25 categories. The ground truth is the category of the resume: a resume is a good result for a job if it belongs to the category of the job. The score is the precision at 5 (the share of the 5 first results of the right category) and the reciprocal rank of the first right result.

In [8]:
job_descriptions = {
    "Data Science": "Data scientist to build machine learning models, analyse large datasets with Python and SQL, and present statistical findings.",
    "HR": "Human resources executive for recruitment, employee relations, payroll, onboarding and HR policies.",
    "Advocate": "Advocate for civil and criminal litigation, legal drafting, court appearances and client advice.",
    "Arts": "Arts graduate for teaching, cultural projects, creative content and community programmes.",
    "Web Designing": "Web designer to build responsive websites with HTML, CSS, JavaScript, Photoshop and user interface design.",
    "Mechanical Engineer": "Mechanical engineer for machine design and manufacturing with AutoCAD, SolidWorks, production and maintenance.",
    "Sales": "Sales executive for business development, client acquisition, sales targets, marketing and customer relationships.",
    "Health and fitness": "Fitness trainer and health coach for exercise programmes, nutrition advice and wellness.",
    "Civil Engineer": "Civil engineer for construction site supervision, structural design, AutoCAD, estimation and project execution.",
    "Java Developer": "Java developer for enterprise applications with Java, Spring, Hibernate, J2EE, web services and SQL.",
    "Business Analyst": "Business analyst to gather requirements, write business documents, model processes and work with stakeholders.",
    "SAP Developer": "SAP consultant with SAP modules, ABAP development, ERP implementation and support.",
    "Automation Testing": "Automation test engineer with Selenium, test scripts, test frameworks and continuous integration.",
    "Electrical Engineering": "Electrical engineer for power systems, electrical design, PLC, circuits, substations and maintenance.",
    "Operations Manager": "Operations manager for supply chain, logistics, process improvement, teams and budgets.",
    "Python Developer": "Python developer for backend applications with Django, Flask, REST APIs and databases.",
    "DevOps Engineer": "DevOps engineer for CI/CD pipelines, Jenkins, Docker, Kubernetes, cloud infrastructure and automation.",
    "Network Security Engineer": "Network security engineer for firewalls, network protocols, Cisco routers, VPN and security monitoring.",
    "PMO": "PMO analyst for project planning, portfolio tracking, reporting, governance and risk management.",
    "Database": "Database administrator for Oracle and SQL Server, backups, performance tuning and data recovery.",
    "Hadoop": "Big data engineer with Hadoop, Hive, Spark, MapReduce and data pipelines.",
    "ETL Developer": "ETL developer for data warehousing, Informatica, data integration and SQL.",
    "DotNet Developer": "DotNet developer with C#, ASP.NET, MVC, web applications and SQL Server.",
    "Blockchain": "Blockchain developer for smart contracts, Ethereum, Solidity and decentralized applications.",
    "Testing": "Software test engineer for manual testing, test cases, defect tracking and quality assurance.",
}

full_vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words="english").fit(unique["cleaned"])
resume_matrix = full_vectorizer.transform(unique["cleaned"])

def search_resumes(job_description, k=5):
    similarity = cosine_similarity(full_vectorizer.transform([clean_resume(job_description)]), resume_matrix).ravel()
    return pd.DataFrame({"resume": np.argsort(-similarity)[:k], "similarity": np.sort(similarity)[::-1][:k]})

def retrieval_scores(ranking):
    relevant = np.array([[unique.loc[r, "Category"] == category for r in ranking[category]] for category in job_descriptions])
    first = np.array([(row.argmax() + 1) if row.any() else np.inf for row in relevant])
    return {"precision at 5": relevant[:, :5].mean(), "mean reciprocal rank": (1 / first).mean()}

rng = np.random.default_rng(42)
rankings = {"TF-IDF search tool": {c: search_resumes(d, 10)["resume"].tolist() for c, d in job_descriptions.items()},
            "random ranking": {c: rng.permutation(len(unique))[:10].tolist() for c in job_descriptions}}
pd.DataFrame({name: retrieval_scores(r) for name, r in rankings.items()}).T.round(3)

,precision at 5,mean reciprocal rank
TF-IDF search tool,0.792,0.980
random ranking,0.016,0.065


The search tool puts a resume of the right category in 79% of the first 5 places (random ranking: 1.6%), and the first result is almost always of the right category (mean reciprocal rank 0.98). It is a strong tool for its simplicity, since the job descriptions and the resumes share many technical words. The category is only a proxy of relevance: a "Database" resume can be a good candidate for a data science job, and it counts as a mistake here, so the true quality of the tool is probably a little higher than these numbers.

## The Screening Agent: Scoring Tool

The second tool asks a language model to read a resume and a job description and answer with a score from 0 to 10 and a one-sentence reason, in JSON format. The model is Llama 3.2 (3 billion parameters), run locally by Ollama, with a temperature of 0 and the first 1,500 characters of each resume. The agent searches the 8 best resumes for a job, scores each of them with the model and ranks them by score (ties are ordered by the similarity of the search tool).

To keep the running time reasonable (each call takes several seconds on a CPU), 12 jobs are tested (96 scored resumes). The scores are saved in `llm_scores.json`, so that the notebook can be run again without calling the model.

In [9]:
import ollama

def score_resume(job_description, resume_text):
    prompt = ("You are an experienced recruiter. Rate how well the resume fits the job, from 0 (not at all) to 10 (perfect fit).\n"
              f"Job: {job_description}\nResume: {resume_text[:1500]}\n"
              'Answer only with JSON: {"score": <number>, "reason": "<one short sentence>"}')
    answer = ollama.chat(model="llama3.2", messages=[{"role": "user", "content": prompt}], format="json", options={"temperature": 0}, keep_alive="30m")
    return json.loads(answer["message"]["content"])

tested_jobs = [c for c in job_descriptions if (unique["Category"] == c).sum() >= 5][:12]
cache_file = "llm_scores.json"
cache = json.load(open(cache_file)) if os.path.exists(cache_file) else {}

for category in tested_jobs:
    for resume in search_resumes(job_descriptions[category], 8)["resume"]:
        key = f"{category}|{resume}"
        if key not in cache:
            cache[key] = score_resume(job_descriptions[category], unique.loc[resume, "Resume"])
            json.dump(cache, open(cache_file, "w"))
print("scored resumes:", len(cache), "for", len(tested_jobs), "jobs")

scored resumes: 96 for 12 jobs


## Evaluation of the Agent

In [10]:
rows = []
for category in tested_jobs:
    found = search_resumes(job_descriptions[category], 8)
    for rank, (resume, similarity) in enumerate(zip(found["resume"], found["similarity"]), start=1):
        answer = cache[f"{category}|{resume}"]
        rows.append({"job": category, "resume": resume, "search rank": rank, "similarity": similarity,
                     "llm score": float(answer.get("score", np.nan)), "relevant": unique.loc[resume, "Category"] == category})
scored = pd.DataFrame(rows)

def ranking_scores(order_column, ascending):
    ranked = scored.sort_values(["job", order_column, "similarity"], ascending=[True, ascending, False])
    ranked["position"] = ranked.groupby("job").cumcount() + 1
    top5 = ranked[ranked["position"] <= 5]
    first = ranked[ranked["relevant"]].groupby("job")["position"].min().reindex(tested_jobs).fillna(np.inf)
    return {"precision at 5": top5["relevant"].mean(), "mean reciprocal rank": (1 / first).mean()}

results = pd.DataFrame({"search tool only (TF-IDF)": ranking_scores("search rank", True), "search + language model score": ranking_scores("llm score", False)}).T
results["AUC of the score (relevant or not)"] = [roc_auc_score(scored["relevant"], -scored["search rank"]), roc_auc_score(scored["relevant"], scored["llm score"].fillna(0))]
results.round(3)

,precision at 5,mean reciprocal rank,AUC of the score (relevant or not)
search tool only (TF-IDF),0.867,1.000,0.800
search + language model score,0.867,0.958,0.715


In [11]:
print("share of the 8 searched resumes that are relevant:", round(scored["relevant"].mean(), 3))
print("mean llm score, relevant resumes:", round(scored[scored["relevant"]]["llm score"].mean(), 2), "| not relevant:", round(scored[~scored["relevant"]]["llm score"].mean(), 2))
scored.groupby("job")[["relevant"]].sum().assign(scored=8).T

share of the 8 searched resumes that are relevant: 0.719
mean llm score, relevant resumes: 4.15 | not relevant: 2.15


job,Advocate,Arts,Automation Testing,Business Analyst,Civil Engineer,Data Science,HR,Health and fitness,Java Developer,Mechanical Engineer,SAP Developer,Sales
relevant,7,4,6,5,6,6,6,6,8,5,6,4
scored,8,8,8,8,8,8,8,8,8,8,8,8


The language model gives a higher score to relevant resumes on average (4.15 against 2.15 out of 10), so its score carries some information. But it does not improve the search ranking: the precision at 5 stays at 0.867 and the mean reciprocal rank goes down (1.000 to 0.958). The AUC, which measures how well a score separates the relevant from the non-relevant resumes among the 8 searched ones, is 0.800 for the search similarity and 0.715 for the model score. A model with 3 billion parameters that reads the first 1,500 characters of a resume gives coarse scores (many resumes get the same score), and its reasons are generic ("lacks relevant experience"). The demonstration for Data Science shows both sides: the resumes of the category get the highest scores (8, 8, 6, 6), but a Database resume also gets 6, and a Data Science resume gets 2. On 12 jobs and 96 resumes, the difference between 0.800 and 0.715 is only suggestive.

A last look at one job, as the agent returns it: the resumes are shown with their number, the category, the score of the language model and its reason. The category is the ground truth and is not given to the model.

In [12]:
demo = "Data Science"
found = search_resumes(job_descriptions[demo], 8)
agent_answer = pd.DataFrame({"resume": found["resume"], "category (hidden from the model)": unique.loc[found["resume"], "Category"].values,
                             "llm score": [cache[f"{demo}|{r}"]["score"] for r in found["resume"]], "reason": [cache[f"{demo}|{r}"].get("reason", "") for r in found["resume"]]})
agent_answer.sort_values("llm score", ascending=False)

,resume,category (hidden from the model),llm score,reason
0,6,Data Science,8,The resume highlights relevant skills and expe...
4,9,Data Science,8,Strong technical skills and experience in mach...
5,134,Database,6,Lacks specific machine learning experience and...
1,8,Data Science,6,The candidate has relevant experience and skil...
6,3,Data Science,6,"The resume highlights relevant skills, but lac..."
2,1,Data Science,4,Lacks specific details about machine learning ...
3,5,Data Science,2,Lacks specific technical experience and skills...
7,136,Hadoop,2,Lacks relevant machine learning and Python exp...


## Conclusion

- The dataset has 962 rows but only 166 different resumes. The reference accuracy of 0.99 comes from copies of the same resume in the training and test sets (99.5% of the test resumes). After removing the copies, the same KNN model reaches 0.735, and Logistic Regression and Naive Bayes reach 0.324 and 0.118.
- A TF-IDF search tool that ranks resumes for a job description puts the right category in 79% of the first 5 results (random: 1.6%) without any training.
- A local language model (Llama 3.2, 3B) scores relevant resumes higher on average (4.15 against 2.15) but does not improve the ranking of the search tool (AUC 0.715 against 0.800). In this setting the language model step adds cost (several seconds per resume on a CPU) without a measured gain, and it should be used, if at all, to explain a ranking rather than to change it.
- The agent is a fixed workflow (search, then score, then rank).
- Limitations: the category is only a proxy for relevance, the 25 job descriptions were written for this notebook, the LLM was evaluated on 12 jobs with 8 resumes each, and the dataset is small; the numbers are a rough guide.